# Feature Engineering & ML Pipeline Architecture

This notebook designs the features required for our predictive models and constructs the `scikit-learn` pipeline architecture. Preprocessing and feature engineering are designed to be fit strictly on training data to prevent target and feature leakage.

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import warnings
warnings.filterwarnings('ignore')

# Add src to path to import our custom transformer
sys.path.append(os.path.abspath('..'))
from src.features.build_features import (
    FeatureEngineer, 
    RAW_NUMERICAL_FEATURES, 
    RAW_CATEGORICAL_FEATURES,
    ENGINEERED_NUMERICAL_FEATURES,
    ENGINEERED_CATEGORICAL_FEATURES
)

df = pd.read_csv('../data/processed/cleaned_telco_churn.csv')
df.head()

## 1. Engineered Features Documentation

All engineered features are strictly derivable from information available *before* the churn event occurs. They are encapsulated in the `FeatureEngineer` class found in `src/features/build_features.py`.

### `service_count`
- **Formula**: Sum of binary flags for [`online_security`, `online_backup`, `device_protection`, `tech_support`, `streaming_tv`, `streaming_movies`]
- **Business Meaning**: Represents the depth of service adoption. High service count may indicate high lock-in.
- **Data Availability**: Known at all times during the subscription.
- **Leakage Risk**: None. Evaluated before churn.
- **Reason for inclusion**: Condenses 6 categorical columns into a single ordinal feature representing ecosystem engagement.

### `is_automatic_payment`
- **Formula**: 1 if `payment_method` contains 'automatic', 0 otherwise.
- **Business Meaning**: Captures payment friction. As seen in EDA, manual payments (checks) drive huge churn.
- **Data Availability**: Set at billing setup.
- **Leakage Risk**: None.
- **Reason for inclusion**: Directly tackles the primary insight from the Payment Method EDA.

### `spend_delta`
- **Formula**: `monthly_charges` - (`total_charges` / `tenure_months`)
- **Business Meaning**: Represents recent price changes. If this is positive, their current bill is higher than their historical average bill (indicating a recent upsell or price hike, which may trigger churn).
- **Data Availability**: Computed from historical billing.
- **Leakage Risk**: None. Depends entirely on current and past billing.
- **Reason for inclusion**: Attempts to capture "bill shock" without needing actual time-series data.

### `has_internet`
- **Formula**: 1 if `internet_service` != 'No', else 0.
- **Business Meaning**: Pure phone vs internet customers.
- **Data Availability**: At subscription.
- **Leakage Risk**: None.
- **Reason for inclusion**: Simple binary flag mapping to our EDA findings on internet vs no-internet retention differences.

## 2. Feature Category Separation

To build a robust pipeline, we must explicitly separate our variables into roles.

In [ ]:
from src.features.build_features import EDA_ONLY_FEATURES, LEAKAGE_FEATURES, TARGET_FEATURE

print(f"Numerical Features (Raw): {RAW_NUMERICAL_FEATURES}")
print(f"Categorical Features (Raw): {RAW_CATEGORICAL_FEATURES}")
print(f"Engineered Numerical: {ENGINEERED_NUMERICAL_FEATURES}")
print(f"Engineered Categorical: {ENGINEERED_CATEGORICAL_FEATURES}")
print(f"EDA Only (Dropped for ML): {EDA_ONLY_FEATURES}")
print(f"Leakage (Strictly Dropped): {LEAKAGE_FEATURES}")
print(f"Target: {TARGET_FEATURE}")

## 3. Testing the Custom Transformer

Let's apply our `FeatureEngineer` to the data and verify the outputs.

In [ ]:
engineer = FeatureEngineer()
df_engineered = engineer.transform(df)

# Check the new features
df_engineered[['service_count', 'is_automatic_payment', 'spend_delta', 'has_internet']].head()

## 4. Pipeline Architecture

We construct the `scikit-learn` `ColumnTransformer` to handle scaling and encoding. 
*Note: We do not `fit()` this pipeline here. It must only be fitted on the `X_train` dataset during the modeling phase to prevent data leakage.*

In [ ]:
# Combine raw and engineered features for the preprocessor
all_numerical = RAW_NUMERICAL_FEATURES + ENGINEERED_NUMERICAL_FEATURES
all_categorical = RAW_CATEGORICAL_FEATURES + ENGINEERED_CATEGORICAL_FEATURES

# Define the ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), all_numerical),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), all_categorical)
    ],
    remainder='drop' # Automatically drops identifiers and leakage columns if they sneak in
)

# Define the full processing pipeline (Model will be appended later)
full_pipeline_blueprint = Pipeline([
    ('feature_engineer', FeatureEngineer()),
    ('preprocessor', preprocessor)
])

print("Pipeline architecture established. Ready for Model Training Phase.")